# TDI Python 2.0 | Week 4: File Handling & Data Quality Assessment

**Project:** Supply Chain Data Quality Assessment  
**Source:** 3,000-row supply chain dataset  
**Tools:** Python standard library (`csv`, `datetime`, `collections`)

Run all cells from top to bottom. Place the original CSV beside this notebook as `supply_chain_dataset.csv`. The raw file is never overwritten.

## Load the raw dataset and define helpers

Counts are computed from the original data. Missing delivery information for non-final orders is treated as expected; all other missing fields are counted.

In [1]:
import csv,collections,datetime,os,json,statistics
from pathlib import Path
p=Path('.') ; src=Path('supply_chain_dataset.csv')
with src.open(newline='',encoding='utf-8-sig') as f:r=csv.DictReader(f); cols=r.fieldnames; rows=list(r)
def dateparse(s):
 if not s.strip():return None
 return datetime.datetime.strptime(s,'%m/%d/%Y' if '/' in s else '%Y-%m-%d').date()
def sup(s):
 s=s.strip().replace(' ','').lower()
 return 'Supplier'+s[-1].upper() if s.startswith('supplier') and s[-1:] in 'abcde' else s
def wh(s):
 s=s.strip()
 return 'Warehouse_'+s[3:] if s.startswith('WH_') else s
def status(s):
 s=s.strip();return {'Delievered':'Delivered','Pendng':'Pending'}.get(s,s)
def normalize(row):
 d={k:v.strip() for k,v in row.items()}
 d['supplier_name']=sup(d['supplier_name']);d['warehouse_destination']=wh(d['warehouse_destination']);d['delivery_status']=status(d['delivery_status'])
 for k in ('order_date','delivery_date'):
  if d[k]:
   try:d[k]=dateparse(d[k]).isoformat()
   except ValueError:pass
 return d


## Task 1: Missing values, duplicates, text variations and date formats

Counts are computed from the original data. Missing delivery information for non-final orders is treated as expected; all other missing fields are counted.

In [2]:
missing={k:sum(not row[k].strip() for row in rows) for k in cols}
seen=set(); duplicate_ids=[]
for row in rows:
    key=tuple(row[k] for k in cols)
    if key in seen: duplicate_ids.append(row['order_id'])
    seen.add(key)
print('Missing cells:',missing)
print('Exact duplicates:',len(duplicate_ids))
for col in ('supplier_name','warehouse_destination','delivery_status','product_category'):
    values=sorted(set(row[col] for row in rows))
    print(f'{col}: {len(values)} distinct:',values)
print('Category cells with surrounding spaces:',sum(row['product_category']!=row['product_category'].strip() for row in rows))
for col in ('order_date','delivery_date'):
    print(col,dict(collections.Counter('blank' if not row[col].strip() else ('slash' if '/' in row[col] else 'ISO') for row in rows)))

Missing cells: {'order_id': 0, 'order_date': 0, 'product_category': 0, 'quantity': 0, 'unit_price': 0, 'total_cost': 0, 'supplier_name': 0, 'supplier_location': 0, 'lead_time_days': 0, 'warehouse_destination': 0, 'delivery_date': 0, 'actual_delivery_days': 0, 'delivery_status': 0, 'on_time': 0, 'product_defects': 154, 'return_status': 146}
Exact duplicates: 0
supplier_name: 15 distinct: ['Supplier A', 'Supplier B', 'Supplier C', 'Supplier D', 'Supplier E', 'SupplierA', 'SupplierB', 'SupplierC', 'SupplierD', 'SupplierE', 'suppliera', 'supplierb', 'supplierc', 'supplierd', 'suppliere']
warehouse_destination: 12 distinct: ['WH_1', 'WH_2', 'WH_3', 'WH_4', 'WH_5', 'WH_6', 'Warehouse_1', 'Warehouse_2', 'Warehouse_3', 'Warehouse_4', 'Warehouse_5', 'Warehouse_6']
delivery_status: 6 distinct: ['Cancelled', 'Delayed', 'Delievered', 'Delivered', 'In-Transit', 'Pendng']
product_category: 8 distinct: ['Connectors', 'Control Systems', 'Fasteners', 'Hydraulics', 'Motors', 'Pumps', 'Sensors', 'Valves'

## Task 2: Validate logical consistency

Counts are computed from the original data. Missing delivery information for non-final orders is treated as expected; all other missing fields are counted.

In [3]:
nonfinal={'Cancelled','In-Transit','Pending','Pendng'}
def validate(row):
 issues=[];id=row['order_id'];s=status(row['delivery_status'])
 for k in cols:
  if not row[k].strip() and not(k=='delivery_date' and s in nonfinal) and not(k=='actual_delivery_days' and s in nonfinal):issues.append('missing_'+k)
 try:
  cost=float(row['total_cost']);qty=float(row['quantity']);unit=float(row['unit_price'])
  if cost<=0:issues.append('nonpositive_cost')
  if abs(cost-qty*unit)>1:issues.append('cost_mismatch')
 except (ValueError,TypeError):issues.append('invalid_cost_or_inputs')
 if s not in nonfinal:
  try:
   order=dateparse(row['order_date']);delivered=dateparse(row['delivery_date'])
   if order and delivered:
    days=(delivered-order).days
    if days<0:issues.append('delivery_before_order')
    try:
     if int(row['actual_delivery_days'])!=days:issues.append('duration_mismatch')
    except ValueError:issues.append('invalid_actual_days')
  except ValueError:issues.append('invalid_delivery_date')
  try:
   actual=int(row['actual_delivery_days']);lead=int(row['lead_time_days']);flag=row['on_time'].strip().lower()
   if flag in ('yes','no') and (flag=='yes')!=(actual<=lead):issues.append('on_time_mismatch')
  except ValueError:issues.append('invalid_lead_or_actual')
 return issues
allissues=[validate(row) for row in rows]; issuecounts=collections.Counter(i for iss in allissues for i in iss)
for i,id in enumerate(duplicate_ids):pass

print('Non-final orders:',sum(status(row['delivery_status']) in nonfinal for row in rows))
for issue,count in sorted(issuecounts.items()):
    examples=[row['order_id'] for row,issues in zip(rows,allissues) if issue in issues][:5]
    print(issue,count,'examples:',examples)

Non-final orders: 107
cost_mismatch 154 examples: ['ORD-0029', 'ORD-0034', 'ORD-0057', 'ORD-0085', 'ORD-0096']
delivery_before_order 23 examples: ['ORD-0007', 'ORD-0060', 'ORD-0122', 'ORD-0331', 'ORD-0408']
duration_mismatch 23 examples: ['ORD-0007', 'ORD-0060', 'ORD-0122', 'ORD-0331', 'ORD-0408']
missing_product_defects 154 examples: ['ORD-0010', 'ORD-0036', 'ORD-0050', 'ORD-0052', 'ORD-0064']
missing_return_status 146 examples: ['ORD-0003', 'ORD-0030', 'ORD-0052', 'ORD-0055', 'ORD-0058']
nonpositive_cost 1 examples: ['ORD-1521']
on_time_mismatch 54 examples: ['ORD-0033', 'ORD-0039', 'ORD-0081', 'ORD-0184', 'ORD-0211']


## Task 3: Standardise, export and verify

Counts are computed from the original data. Missing delivery information for non-final orders is treated as expected; all other missing fields are counted.

In [4]:
changed=collections.Counter();new=[]
for row in rows:
 d=normalize(row);new.append(d)
 for k in cols:
  if d[k]!=row[k]:changed[k]+=1
with (p/'supply_chain_standardised.csv').open('w',newline='',encoding='utf-8') as f:
 w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(new)
with (p/'supply_chain_standardised.csv').open(newline='',encoding='utf-8') as f:verified=list(csv.DictReader(f))
assert len(verified)==len(rows)
by_supplier=collections.Counter();by_month=collections.Counter()
for row,issues in zip(rows,allissues):
 if issues:
  by_supplier[sup(row['supplier_name'])]+=1
  try:by_month[dateparse(row['order_date']).strftime('%Y-%m')]+=1
  except:pass

print('Changes by column:',dict(changed))
for col in ('supplier_name','warehouse_destination','delivery_status'):
    print(col,'after:',sorted(set(row[col] for row in verified)))

Changes by column: {'order_date': 3000, 'delivery_date': 3000, 'supplier_name': 1945, 'warehouse_destination': 276, 'delivery_status': 43}
supplier_name after: ['SupplierA', 'SupplierB', 'SupplierC', 'SupplierD', 'SupplierE']
warehouse_destination after: ['Warehouse_1', 'Warehouse_2', 'Warehouse_3', 'Warehouse_4', 'Warehouse_5', 'Warehouse_6']
delivery_status after: ['Cancelled', 'Delayed', 'Delivered', 'In-Transit', 'Pending']


## Extra challenges: Validation, suppliers and monthly patterns

Counts are computed from the original data. Missing delivery information for non-final orders is treated as expected; all other missing fields are counted.

In [5]:
print('Challenge 1: rows failing validate_order:',sum(bool(issues) for issues in allissues))
by_supplier=collections.Counter();by_month=collections.Counter()
for row,issues in zip(rows,allissues):
    if issues:
        by_supplier[sup(row['supplier_name'])]+=1
        by_month[dateparse(row['order_date']).strftime('%Y-%m')]+=1
print('Challenge 2: problem rows by supplier:',by_supplier)
print('Challenge 3: top months:',by_month.most_common(10))

Challenge 1: rows failing validate_order: 498
Challenge 2: problem rows by supplier: Counter({'SupplierA': 123, 'SupplierE': 114, 'SupplierB': 106, 'SupplierC': 80, 'SupplierD': 75})
Challenge 3: top months: [('2022-05', 31), ('2022-03', 29), ('2023-04', 28), ('2023-12', 28), ('2023-10', 26), ('2022-06', 25), ('2022-08', 23), ('2022-02', 22), ('2023-01', 22), ('2022-10', 22)]


## Task 4: Data quality report

| Issue | Rows affected | Fixed this week? | What should happen next |
|---|---:|---|---|
| Missing product_defects | 154 | No | Investigate missingness; choose treatment in Week 7 |
| Missing return_status | 146 | No | Check whether blanks mean no return or unknown |
| Exact duplicates | 0 | Not needed | Continue monitoring |
| Cost mismatches (>1) | 154 | No | Review source values and recalculate only with evidence |
| Zero/negative costs | 1 | No | Investigate invalid cost |
| Delivery before order | 23 | No | Verify order and delivery dates |
| Actual duration mismatch | 23 | No | Reconcile with date difference |
| On-time mismatch | 54 | No | Review on-time rule and source flags |
| Supplier names changed | 1,945 | Yes | Keep canonical names |
| Warehouse names changed | 276 | Yes | Keep canonical names |
| Delivery statuses changed | 43 | Yes | Use controlled vocabulary |
| Order dates converted | 3,000 | Yes | Keep ISO dates |
| Delivery dates converted | 3,000 | Yes | Keep ISO dates |

**Before/after summary:** The audit flagged 498 orders with at least one missing or logical issue (16.6%). The most frequent single issue was missing product defects (154), tied with cost mismatches (154). Standardisation changed supplier names in 1,945 rows, warehouse names in 276, status labels in 43, and both date fields in all 3,000 rows. No records were deleted and missing values or numerical inconsistencies remain for later investigation. The issue counts overlap and should not be added together as unique orders.

**Caution:** This assessment excludes expected missing delivery fields for non-final statuses. The `validate_order` function flags non-final rows only for other applicable problems.
